In [2]:
# =============================================================================
# NASA JPL Fireball Data API - Complete Tutorial
# =============================================================================
# This notebook shows how to fetch fireball/bolide data from the CNEOS
# (Center for Near-Earth Object Studies) Fireball API.
# API documentation: https://ssd-api.jpl.nasa.gov/doc/fireball.html
# =============================================================================

# Tutorial: NASA JPL Fireball Data API (CNEOS)
# Required libraries
import requests
import pandas as pd
import json
from datetime import datetime

# Fireball Data API – Full Tutorial

This notebook is a **complete tutorial** for the [NASA JPL Fireball Data API](https://ssd-api.jpl.nasa.gov/doc/fireball.html), which provides access to the **CNEOS Fireball** dataset (bright meteors / fireballs detected by sensors).

- **Base URL:** `https://ssd-api.jpl.nasa.gov/fireball.api`
- **Method:** `GET`
- **Output:** JSON (list of fireball records with configurable fields)

We will cover every **query parameter**, **response structure**, and **data field**, with runnable examples.

---

## 1. Basic request and response structure

**Input:** A GET request with optional query parameters.  
**Output:** A JSON object with:
- `signature`: API version and source (always check version matches [documentation](https://ssd-api.jpl.nasa.gov/doc/fireball.html)).
- `count`: Number of records returned.
- `fields`: List of field names for each record.
- `data`: List of records; each record is an array of values in the same order as `fields`.

If no parameters are given, the API returns **all** fireball data in **reverse chronological order** (newest first).

In [33]:
# Simplest call: no parameters → all data, newest first
BASE_URL = "https://ssd-api.jpl.nasa.gov/fireball.api"
response = requests.get(BASE_URL)

# Inspect response structure
data = response.json()
print("Keys in response:", list(data.keys()))
print("API version:", data["signature"].get("version"))
print("Number of records:", data["count"])
print("Field names:", data["fields"])

Keys in response: ['signature', 'count', 'fields', 'data']
API version: 1.2
Number of records: 1052
Field names: ['date', 'energy', 'impact-e', 'lat', 'lat-dir', 'lon', 'lon-dir', 'alt', 'vel']


In [34]:
# One record is a list of values; order matches data["fields"]
print("Example record (first fireball):")
first_record = data["data"][0]
for name, value in zip(data["fields"], first_record):
    print(f"  {name}: {value}")

Example record (first fireball):
  date: 2026-02-10 14:26:26
  energy: 2.1
  impact-e: 0.076
  lat: 64.0
  lat-dir: S
  lon: 14.0
  lon-dir: W
  alt: 44.0
  vel: None


---

## 2. Query parameters (filters, selectors, sort, limit)

| Parameter       | Type    | Default  | Role     | Description |
|----------------|---------|----------|----------|-------------|
| **date-min**   | string  | none     | filter   | Exclude data *earlier than* this date (`YYYY-MM-DD` or `YYYY-MM-DDThh:mm:ss`) |
| **date-max**   | string  | none     | filter   | Exclude data *later than* this date |
| **energy-min** | string  | none     | filter   | Exclude radiated energy *less than* this (in 10¹⁰ J) |
| **energy-max** | string  | none     | filter   | Exclude radiated energy *greater than* this |
| **impact-e-min** | string | none   | filter   | Exclude impact energy *less than* this (kilotons, kt) |
| **impact-e-max** | string | none   | filter   | Exclude impact energy *greater than* this (kt) |
| **alt-min**    | number  | none     | filter   | Exclude objects with altitude *less than* this (km) |
| **alt-max**    | number  | none     | filter   | Exclude objects with altitude *greater than* this (km) |
| **req-loc**    | boolean | false    | filter   | If true, only records with latitude/longitude |
| **req-alt**    | boolean | false    | filter   | If true, only records with altitude |
| **req-vel-comp** | boolean | false  | filter   | If true, only records with entry velocity components (vx, vy, vz) |
| **vel-comp**   | boolean | false    | selector | If true, include vx, vy, vz in the response |
| **sort**       | string  | "-date"  | sorter   | Sort by: "date", "energy", "impact-e", "vel", "alt"; prefix "-" for descending |
| **limit**      | number  | none     | filter   | Return only the first N results (integer > 0) |

In [35]:
# --- Example 2.1: limit ---
# Return only the first N records (useful for testing or recent events)
r = requests.get(BASE_URL, params={"limit": 5})
print("limit=5 →", r.json()["count"], "records")
pd.DataFrame(r.json()["data"], columns=r.json()["fields"])

limit=5 → 5 records


,date,energy,impact-e,lat,lat-dir,lon,lon-dir,alt,vel
0,2026-02-10 14:26:26,2.1,0.076,64.0,S,14.0,W,44.0,None
1,2026-01-31 18:07:14,5.8,0.19,4.1,N,173.4,W,32.0,None
2,2026-01-30 10:25:37,3.4,0.12,45.0,S,174.5,E,89.0,71.1
3,2025-12-16 20:58:12,9.6,0.29,24.1,S,92.4,W,25.0,19.0
4,2025-11-15 00:48:43,10.5,0.32,62.2,S,94.7,W,30.0,16.0


### 2.2 date-min and date-max

Filter by **date of peak brightness** (GMT). Format: `YYYY-MM-DD` or `YYYY-MM-DDThh:mm:ss`.
- **date-min**: exclude records *earlier than* this date.
- **date-max**: exclude records *later than* this date.

In [36]:
r = requests.get(BASE_URL, params={
    "date-min": "2025-01-01",
    "date-max": "2025-02-01",
    "limit": 10
})
j = r.json()
print("Fireballs between 2025-01-01 and 2025-02-01:", j["count"], "records")
pd.DataFrame(j["data"], columns=j["fields"])

Fireballs between 2025-01-01 and 2025-02-01: 2 records


,date,energy,impact-e,lat,lat-dir,lon,lon-dir,alt,vel
0,2025-01-11 14:46:00,2.9,0.1,1.2,S,55.3,E,61.0,13.2
1,2025-01-10 21:11:07,2.7,0.095,47.2,N,106.3,E,34.6,14.0


### 2.3 energy-min and energy-max

Filter by **total radiated energy** in units of 10¹⁰ joules (e.g. `0.3` = 0.3×10¹⁰ J).

In [ ]:
r = requests.get(BASE_URL, params={
    "energy-min": "1",
    "energy-max": "10",
    "limit": 5
})
j = r.json()
print("Radiated energy between 1e10 and 10e10 J:", j["count"], "records")
pd.DataFrame(j["data"], columns=j["fields"])

NameError: name 'requests' is not defined

### 2.4 impact-e-min and impact-e-max

Filter by **impact energy** in **kilotons (kt)** (e.g. `0.08` = 0.08 kt).

In [38]:
r = requests.get(BASE_URL, params={
    "impact-e-min": "0.1",
    "impact-e-max": "0.5",
    "limit": 5
})
j = r.json()
print("Impact energy between 0.1 and 0.5 kt:", j["count"], "records")
pd.DataFrame(j["data"], columns=j["fields"])

Impact energy between 0.1 and 0.5 kt: 5 records


,date,energy,impact-e,lat,lat-dir,lon,lon-dir,alt,vel
0,2026-01-31 18:07:14,5.8,0.19,4.1,N,173.4,W,32.0,None
1,2026-01-30 10:25:37,3.4,0.12,45.0,S,174.5,E,89.0,71.1
2,2025-12-16 20:58:12,9.6,0.29,24.1,S,92.4,W,25.0,19.0
3,2025-11-15 00:48:43,10.5,0.32,62.2,S,94.7,W,30.0,16.0
4,2025-11-11 17:39:51,9.3,0.28,27.3,N,79.8,W,42.0,18.7


### 2.5 alt-min and alt-max

Filter by **altitude above geoid at peak brightness** (km). Smaller numeric altitude = larger object.
- **alt-min**: exclude objects with altitude *less than* this (i.e. smaller than this).
- **alt-max**: exclude objects with altitude *greater than* this (i.e. larger than this).

In [ ]:
# req-alt=true ensures we only get records that have altitude (so alt filters apply)
r = requests.get(BASE_URL, params={
    "alt-min": 10,
    "alt-max": 90
})  #revisar por que retorna 500 la API
j = r.json()
print("Altitude between 20 and 50 km:", j["count"], "records")
pd.DataFrame(j["data"], columns=j["fields"])

KeyError: 'count'

### 2.6 req-loc, req-alt, req-vel-comp (required fields)

- **req-loc** = true → only records with latitude and longitude.
- **req-alt** = true → only records with altitude.
- **req-vel-comp** = true → only records with entry velocity components (vx, vy, vz).

In [49]:
r

<Response [500]>

In [40]:
# Only events with location AND altitude (good for mapping)
r = requests.get(BASE_URL, params={"req-loc": True, "req-alt": True, "limit": 5})
j = r.json()
print("With location and altitude:", j["count"], "total; showing 5")
pd.DataFrame(j["data"], columns=j["fields"])

With location and altitude: 5 total; showing 5


,date,energy,impact-e,lat,lat-dir,lon,lon-dir,alt,vel
0,2026-02-10 14:26:26,2.1,0.076,64.0,S,14.0,W,44.0,None
1,2026-01-31 18:07:14,5.8,0.19,4.1,N,173.4,W,32.0,None
2,2026-01-30 10:25:37,3.4,0.12,45.0,S,174.5,E,89.0,71.1
3,2025-12-16 20:58:12,9.6,0.29,24.1,S,92.4,W,25.0,19.0
4,2025-11-15 00:48:43,10.5,0.32,62.2,S,94.7,W,30.0,16.0


### 2.7 vel-comp (include velocity components)

**vel-comp** = true adds **vx, vy, vz** (Earth-centered entry velocity components, km/s) to the response. Use **req-vel-comp** = true if you want only records that have these components.

In [41]:
r = requests.get(BASE_URL, params={
    "vel-comp": True,
    "req-vel-comp": True,
    "limit": 5
})
j = r.json()
print("Fields when vel-comp=true:", j["fields"])
pd.DataFrame(j["data"], columns=j["fields"])

Fields when vel-comp=true: ['date', 'energy', 'impact-e', 'lat', 'lat-dir', 'lon', 'lon-dir', 'alt', 'vel', 'vx', 'vy', 'vz']


,date,energy,impact-e,lat,lat-dir,lon,lon-dir,alt,vel,vx,vy,vz
0,2026-01-30 10:25:37,3.4,0.12,45.0,S,174.5,E,89.0,71.1,-2.1,68.8,17.7
1,2025-12-16 20:58:12,9.6,0.29,24.1,S,92.4,W,25.0,19.0,7.0,13.9,10.9
2,2025-11-15 00:48:43,10.5,0.32,62.2,S,94.7,W,30.0,16.0,9.7,2.4,12.5
3,2025-11-11 17:39:51,9.3,0.28,27.3,N,79.8,W,42.0,18.7,-8.0,10.0,-13.6
4,2025-10-20 13:31:27,2.0,0.073,5.7,N,135.2,W,41.0,26.7,16.7,6.0,19.9


### 2.8 sort

Sort by: **date**, **energy**, **impact-e**, **vel**, **alt**. Default is **-date** (newest first). Prepend **-** for descending (e.g. **-energy** = highest energy first).

In [42]:
# Highest impact energy first
r = requests.get(BASE_URL, params={"sort": "-impact-e", "limit": 5})
j = r.json()
pd.DataFrame(j["data"], columns=j["fields"])

,date,energy,impact-e,lat,lat-dir,lon,lon-dir,alt,vel
0,2013-02-15 03:20:26,37500,441,54.8,N,61.1,E,23.3,18.6
1,2018-12-18 23:48:18,3130,49,56.9,N,172.4,E,26.0,13.6
2,2009-10-08 02:57:14,1980,33,4.2,S,120.6,E,19.1,19.2
3,1994-02-01 22:38:09,1820.0,30,2.7,N,164.1,E,None,None
4,2009-11-21 20:53:29,1120,20,22.0,S,29.2,E,38.0,32.1


---

## 3. Data fields (output columns)

Each record in **data** is an array; the **fields** array gives the name for each position. Many fields can be **null**. Only **date**, **energy**, and **impact-e** are guaranteed.

| Field    | Description |
|----------|-------------|
| **date** | Date/time of peak brightness (GMT), `YYYY-MM-DD hh:mm:ss` |
| **lat**  | Latitude at peak brightness (decimal degrees) |
| **lon**  | Longitude at peak brightness (decimal degrees) |
| **lat-dir** | "N" or "S" |
| **lon-dir** | "E" or "W" |
| **alt**  | Altitude above geoid at peak brightness (km) |
| **energy** | Total radiated energy in 10¹⁰ joules |
| **impact-e** | Impact energy (kilotons, kt) |
| **vx, vy, vz** | Entry velocity, Earth-centered components (km/s); only if **vel-comp**=true |

---

## 4. Building a pandas DataFrame

Use the **fields** and **data** from any response to build a DataFrame. Always check **count** and **signature.version** when relying on the API.

In [43]:
def fireball_response_to_dataframe(response):
    """Turn a fireball API response into a pandas DataFrame."""
    j = response.json()
    if int(j.get("count", 0)) == 0:
        return pd.DataFrame(columns=j.get("fields", []))
    return pd.DataFrame(j["data"], columns=j["fields"])

# Example: full dataset with velocity components (may take a moment)
response = requests.get(BASE_URL, params={"vel-comp": True})
df_fireballs = fireball_response_to_dataframe(response)
print("API version:", response.json()["signature"]["version"])
print("Total records:", len(df_fireballs))
df_fireballs.head(10)

API version: 1.2
Total records: 1052


,date,energy,impact-e,lat,lat-dir,lon,lon-dir,alt,vel,vx,vy,vz
0,2026-02-10 14:26:26,2.1,0.076,64.0,S,14.0,W,44.0,None,None,None,None
1,2026-01-31 18:07:14,5.8,0.19,4.1,N,173.4,W,32.0,None,None,None,None
2,2026-01-30 10:25:37,3.4,0.12,45.0,S,174.5,E,89.0,71.1,-2.1,68.8,17.7
3,2025-12-16 20:58:12,9.6,0.29,24.1,S,92.4,W,25.0,19.0,7.0,13.9,10.9
4,2025-11-15 00:48:43,10.5,0.32,62.2,S,94.7,W,30.0,16.0,9.7,2.4,12.5
5,2025-11-11 17:39:51,9.3,0.28,27.3,N,79.8,W,42.0,18.7,-8.0,10.0,-13.6
6,2025-10-20 13:31:27,2.0,0.073,5.7,N,135.2,W,41.0,26.7,16.7,6.0,19.9
7,2025-09-13 22:24:59,16.7,0.48,38.1,S,64.8,W,25.1,17.8,-17.4,1.7,3.6
8,2025-09-10 11:00:57,2.8,0.098,14.8,S,142.4,W,37.0,None,None,None,None
9,2025-09-09 17:49:10,15.3,0.44,2.3,S,39.5,W,24.0,20.7,-3.6,20.4,0.0


---

## 5. Combined example: filters + sort + limit

Example: fireballs from 2014 onward, with location and altitude, sorted by impact energy (highest first), limit 10.

In [44]:
params = {
    "date-min": "2014-01-01",
    "req-loc": True,
    "req-alt": True,
    "sort": "-impact-e",
    "limit": 10
}
r = requests.get(BASE_URL, params=params)
j = r.json()
print("Count:", j["count"], "| Version:", j["signature"]["version"])
pd.DataFrame(j["data"], columns=j["fields"])

Count: 10 | Version: 1.2


,date,energy,impact-e,lat,lat-dir,lon,lon-dir,alt,vel
0,2018-12-18 23:48:18,3130,49,56.9,N,172.4,E,26.0,13.6
1,2016-02-06 13:55:09,685,13,30.4,S,25.5,W,31.0,15.6
2,2020-12-22 23:23:28,667,12,31.9,N,96.2,E,35.5,13.6
3,2017-12-15 13:14:37,400,7.9,60.2,N,170.0,E,20.0,31.4
4,2014-08-23 06:29:41,382,7.6,61.7,S,132.6,E,22.2,16.2
5,2023-05-20 11:22:22,351,7.1,17.8,S,141.9,E,29.0,27.9
6,2022-02-07 20:06:25,348,7,28.7,S,11.4,E,26.5,13.1
7,2023-04-15 08:21:58,321,6.5,20.1,S,36.0,E,41.4,17.2
8,2019-06-22 21:25:47,294,6,14.9,N,66.2,W,25.0,14.9
9,2024-07-20 14:08:10,243.3,5.1,73.7,S,22.5,E,38.8,30.8


---

## 6. Error handling and HTTP response codes

Always check `response.status_code`. The API uses standard HTTP codes:

| Code | Meaning | When it happens |
|------|---------|------------------|
| **200** | OK | Data returned successfully |
| **400** | Bad Request | Invalid parameters or keywords |
| **405** | Method Not Allowed | Wrong HTTP method (must be GET) |
| **500** | Internal Server Error | Database unavailable |
| **503** | Service Unavailable | Server overloaded or in maintenance |

When constraints match no records, you still get **200** but with `"count": 0` and no `"data"` (or empty).

In [45]:
# Safe request helper
def fetch_fireballs(**params):
    r = requests.get(BASE_URL, params=params)
    if r.status_code != 200:
        raise RuntimeError(f"API error {r.status_code}: {r.reason}")
    j = r.json()
    version = j.get("signature", {}).get("version", "?")
    if version != "1.2":
        print(f"Warning: API version is {version}; document assumes 1.2")
    return j

# Example: no results (still 200) — use dict for params with hyphens
j = fetch_fireballs(**{"date-min": "2030-01-01", "limit": 5})
print("Count:", j["count"], "→ empty or no 'data' key")
print("Keys:", list(j.keys()))

Count: 0 → empty or no 'data' key
Keys: ['signature', 'count']


In [46]:
# Optional: one more combined query (date + energy range)
# Note: lat/lon are not filter parameters in the API; filter by date, energy, impact-e, alt, and req-*.
r = requests.get(BASE_URL, params={
    "date-min": "2025-01-01",
    "date-max": "2025-02-02",
    "energy-min": "0.1",
    "energy-max": "5",
    "req-loc": True,
    "limit": 10
})
r.json()

{'signature': {'version': '1.2', 'source': 'NASA/JPL Fireball Data API'},
 'count': '2',
 'fields': ['date',
  'energy',
  'impact-e',
  'lat',
  'lat-dir',
  'lon',
  'lon-dir',
  'alt',
  'vel'],
 'data': [['2025-01-11 14:46:00',
   '2.9',
   '0.1',
   '1.2',
   'S',
   '55.3',
   'E',
   '61.0',
   '13.2'],
  ['2025-01-10 21:11:07',
   '2.7',
   '0.095',
   '47.2',
   'N',
   '106.3',
   'E',
   '34.6',
   '14.0']]}

In [47]:
# Show count and first record from the query above
j = r.json()
print("Count:", j["count"])
if j.get("data"):
    print("First record:", dict(zip(j["fields"], j["data"][0])))

Count: 2
First record: {'date': '2025-01-11 14:46:00', 'energy': '2.9', 'impact-e': '0.1', 'lat': '1.2', 'lat-dir': 'S', 'lon': '55.3', 'lon-dir': 'E', 'alt': '61.0', 'vel': '13.2'}


In [ ]:
# --- References ---
# API doc: https://ssd-api.jpl.nasa.gov/doc/fireball.html
# CNEOS Fireballs: https://cneos.jpl.nasa.gov/fireballs/

# Near Earth Asteroids (NEA) Data API – Full Tutorial

This section covers the [SBDB Query API](https://ssd-api.jpl.nasa.gov/doc/sbdb_query.html) for querying **asteroids and comets** from JPL's Small-Body Database (SBDB), including **Near-Earth Objects (NEOs)** and **Potentially Hazardous Asteroids (PHAs)**.

- **Base URL:** `https://ssd-api.jpl.nasa.gov/sbdb_query.api`
- **Method:** `GET`
- **Output:** JSON (signature, fields, data)

We cover **info mode**, **filter parameters** (`sb-class`, `sb-kind`, `sb-group`, etc.), **output parameters** (`fields`, `sort`, `limit`, `limit-from`, `full-prec`), and **combined examples**.

In [3]:
# Base URL for SBDB Query API
SBDB_URL = "https://ssd-api.jpl.nasa.gov/sbdb_query.api"

# Basic NEO query: orbital elements for Near-Earth Asteroids
params = {
    "fields": "full_name,epoch,a,e,i,om,w,ma,q",
    "sb-kind": "a",    # a=asteroids, c=comets
    "sb-group": "neo"  # neo=NEOs, pha=Potentially Hazardous Asteroids
}
r = requests.get(SBDB_URL, params=params)
data = r.json()
print("API version:", data["signature"]["version"])
print("Fields:", data["fields"])
print("Records returned:", len(data.get("data", [])))

API version: 1.0
Fields: ['full_name', 'epoch', 'a', 'e', 'i', 'om', 'w', 'ma', 'q']
Records returned: 41075


In [4]:
data

{'signature': {'version': '1.0',
  'source': 'NASA/JPL SBDB (Small-Body DataBase) Query API'},
 'fields': ['full_name', 'epoch', 'a', 'e', 'i', 'om', 'w', 'ma', 'q'],
 'data': [['   433 Eros (A898 PA)',
   '2461000.5',
   '1.458',
   '0.2228',
   '10.83',
   '304.27',
   '178.93',
   '310.55',
   '1.133'],
  ['   719 Albert (A911 TB)',
   '2461000.5',
   '2.637',
   '0.5466',
   '11.57',
   '183.86',
   '156.19',
   '240.61',
   '1.195'],
  ['   887 Alinda (A918 AA)',
   '2461000.5',
   '2.474',
   '0.5712',
   '9.40',
   '110.41',
   '350.53',
   '81.54',
   '1.061'],
  ['  1036 Ganymed (A924 UB)',
   '2461000.5',
   '2.665',
   '0.5332',
   '26.68',
   '215.44',
   '132.50',
   '97.59',
   '1.244'],
  ['  1221 Amor (1932 EA1)',
   '2461000.5',
   '1.92',
   '0.4346',
   '11.87',
   '171.24',
   '26.76',
   '59.87',
   '1.085'],
  ['  1566 Icarus (1949 MA)',
   '2461000.5',
   '1.078',
   '0.8270',
   '22.80',
   '87.95',
   '31.44',
   '153.08',
   '0.186'],
  ['  1580 Betulia (1950 

In [9]:
df = pd.DataFrame(data['data'], columns = data['fields'])

In [10]:
df

,full_name,epoch,a,e,i,om,w,ma,q
0,433 Eros (A898 PA),2461000.5,1.458,0.2228,10.83,304.27,178.93,310.55,1.133
1,719 Albert (A911 TB),2461000.5,2.637,0.5466,11.57,183.86,156.19,240.61,1.195
2,887 Alinda (A918 AA),2461000.5,2.474,0.5712,9.40,110.41,350.53,81.54,1.061
3,1036 Ganymed (A924 UB),2461000.5,2.665,0.5332,26.68,215.44,132.50,97.59,1.244
4,1221 Amor (1932 EA1),2461000.5,1.92,0.4346,11.87,171.24,26.76,59.87,1.085
...,...,...,...,...,...,...,...,...,...
41070,(2026 EJ),2461000.5,1.004,0.3428,9.37,346.00,67.33,329.45,0.660
41071,(2026 EK),2461000.5,1.061,0.4565,0.82,140.91,136.90,206.35,0.577
41072,(2026 EL),2461000.5,1.92,0.4687,1.02,22.98,142.54,320.77,1.020
41073,(2026 EM),2461000.5,1.108,0.1921,4.77,346.88,247.16,221.61,0.895


In [ ]:
# Get total counts by small-body type
r = requests.get(SBDB_URL, params={"info": "count"})
j = r.json()
print("Count info:", j)

Count info: {'signature': {'version': '1.0', 'source': 'NASA/JPL SBDB (Small-Body DataBase) Query API'}, 'info': {'count': {'an': 875150, 'cn': 607, 'cu': 3448, 'au': 644867}}}


---

## 2. Filter parameters (sb-*)

All filters use the **sb-** prefix. Combine them to narrow results.

| Parameter | Values | Description |
|-----------|--------|--------------|
| **sb-kind** | `a`, `c` | Asteroids only (`a`) or comets only (`c`) |
| **sb-group** | `neo`, `pha` | NEOs only or PHAs (Potentially Hazardous) only |
| **sb-class** | `IEO`, `ATE`, `APO`, `AMO`, `MBA`, `CEN`, `TNO`, etc. | Orbit class (comma-separated for multiple) |
| **sb-ns** | `n`, `u` | Numbered (`n`) or unnumbered (`u`) only |
| **sb-xfrag** | `true`/`1`, `false`/`0` | Exclude comet fragments |
| **sb-sat** | `true`/`1` | Only objects with known satellites |

In [ ]:
# Atira-class NEAs (interior to Earth's orbit)
r = requests.get(SBDB_URL, params={
    "fields": "full_name,a,e,q",
    "sb-class": "IEO",
    "sb-kind": "a",
    "limit": 5
})
pd.DataFrame(r.json()["data"], columns=r.json()["fields"])

,full_name,a,e,q
0,163693 Atira (2003 CP20),0.741,0.3221,0.502
1,164294 (2004 XZ130),0.6176,0.4545,0.337
2,413563 (2005 TG45),0.6813,0.3723,0.428
3,418265 (2008 EA32),0.6159,0.3050,0.428
4,434326 (2004 JG6),0.6352,0.5311,0.298


In [ ]:
# Apollo-class NEAs (Earth-crossing)
r = requests.get(SBDB_URL, params={
    "fields": "full_name,a,e,q",
    "sb-class": "APO",
    "sb-kind": "a",
    "limit": 5
})
pd.DataFrame(r.json()["data"], columns=r.json()["fields"])

,full_name,a,e,q
0,1566 Icarus (1949 MA),1.078,0.8270,0.186
1,1620 Geographos (1951 RA),1.246,0.3355,0.828
2,1685 Toro (1948 OA),1.368,0.4360,0.771
3,1862 Apollo (1932 HA),1.471,0.5599,0.647
4,1863 Antinous (1948 EA),2.26,0.6063,0.890


### 2.2 sb-group (neo, pha)

- **neo** → Near-Earth Objects only
- **pha** → Potentially Hazardous Asteroids only (subset of NEOs)

In [ ]:
# Potentially Hazardous Asteroids (PHAs)
r = requests.get(SBDB_URL, params={
    "fields": "full_name,a,e,q,i",
    "sb-group": "pha",
    "sb-kind": "a",
    "limit": 5
})
pd.DataFrame(r.json()["data"], columns=r.json()["fields"])

,full_name,a,e,q,i
0,1566 Icarus (1949 MA),1.078,0.8270,0.186,22.80
1,1620 Geographos (1951 RA),1.246,0.3355,0.828,13.34
2,1862 Apollo (1932 HA),1.471,0.5599,0.647,6.35
3,1981 Midas (1973 EA),1.776,0.6505,0.621,39.82
4,2101 Adonis (1936 CA),1.874,0.7641,0.442,1.32


### 2.3 sb-ns (numbered vs unnumbered)

- **sb-ns=n** → numbered objects only (have permanent numbers)
- **sb-ns=u** → unnumbered only

In [ ]:
# Numbered NEOs only
r = requests.get(SBDB_URL, params={
    "fields": "full_name,a,e",
    "sb-group": "neo",
    "sb-ns": "n",
    "limit": 5
})
pd.DataFrame(r.json()["data"], columns=r.json()["fields"])

,full_name,a,e
0,433 Eros (A898 PA),1.458,0.2228
1,719 Albert (A911 TB),2.637,0.5466
2,887 Alinda (A918 AA),2.474,0.5712
3,1036 Ganymed (A924 UB),2.665,0.5332
4,1221 Amor (1932 EA1),1.92,0.4346


---

## 3. Output parameters (fields, sort, limit, limit-from, full-prec)

| Parameter | Description |
|-----------|-------------|
| **fields** | Comma-separated list of output fields (case-sensitive). If omitted, only count is returned. |
| **sort** | Sort by up to 3 fields; prefix `-` for descending (e.g. `-a` = semi-major axis descending) |
| **limit** | Return only the first N records (integer > 0) |
| **limit-from** | Start from record N (for pagination); requires `limit` |
| **full-prec** | `true`/`1` for full precision; default is reduced precision for display |

### Common orbital element fields

| Field | Description | Unit |
|-------|-------------|------|
| **full_name** | Object designation | - |
| **epoch** | Epoch of orbital solution | JD |
| **a** | Semi-major axis | au |
| **e** | Eccentricity | - |
| **i** | Inclination | deg |
| **om** | Longitude of ascending node | deg |
| **w** | Argument of perihelion | deg |
| **ma** | Mean anomaly | deg |
| **q** | Perihelion distance | au |
| **Q** | Aphelion distance | au |

In [ ]:
# Common orbital element fields: full_name, epoch, a, e, i, om, w, ma, q, Q, n, tp, per, M
# Sort by semi-major axis descending (largest first), limit 5
r = requests.get(SBDB_URL, params={
    "fields": "full_name,a,e,i",
    "sb-group": "neo",
    "sb-kind": "a",
    "sort": "-a",
    "limit": 5
})
pd.DataFrame(r.json()["data"], columns=r.json()["fields"])

,full_name,a,e,i
0,(2017 UR52),350.3,0.9964,108.26
1,(A/2024 G8),144.3,0.9919,97.41
2,(2016 XK24),132.5,0.9904,145.63
3,(2019 EJ3),96.58,0.9888,139.98
4,(A/2019 Q2),59.68,0.9789,159.03


In [ ]:
# Pagination: limit=3, limit-from=5 (skip first 5, return next 3)
r = requests.get(SBDB_URL, params={
    "fields": "full_name,a,e",
    "sb-group": "neo",
    "sb-kind": "a",
    "sort": "a",
    "limit": 3,
    "limit-from": 5
})
pd.DataFrame(r.json()["data"], columns=r.json()["fields"])

,full_name,a,e
0,(2019 LF6),0.5554,0.4293
1,(2019 AQ3),0.5886,0.3143
2,(2021 BS1),0.5984,0.3376


---

## 4. Combined examples and DataFrame

Combine filters and output parameters for custom queries.

In [ ]:
# Helper: SBDB response to DataFrame
def sbdb_to_dataframe(response):
    j = response.json()
    if "data" not in j or not j["data"]:
        return pd.DataFrame(columns=j.get("fields", []))
    return pd.DataFrame(j["data"], columns=j["fields"])

# Full NEO orbital elements as DataFrame
r = requests.get(SBDB_URL, params={
    "fields": "full_name,epoch,a,e,i,om,w,ma,q",
    "sb-group": "neo",
    "sb-kind": "a",
    "limit": 20
})
df_neo = sbdb_to_dataframe(r)
df_neo.head(10)

,full_name,epoch,a,e,i,om,w,ma,q
0,433 Eros (A898 PA),2461000.5,1.458,0.2228,10.83,304.27,178.93,310.55,1.133
1,719 Albert (A911 TB),2461000.5,2.637,0.5466,11.57,183.86,156.19,240.61,1.195
2,887 Alinda (A918 AA),2461000.5,2.474,0.5712,9.40,110.41,350.53,81.54,1.061
3,1036 Ganymed (A924 UB),2461000.5,2.665,0.5332,26.68,215.44,132.50,97.59,1.244
4,1221 Amor (1932 EA1),2461000.5,1.92,0.4346,11.87,171.24,26.76,59.87,1.085
5,1566 Icarus (1949 MA),2461000.5,1.078,0.8270,22.80,87.95,31.44,153.08,0.186
6,1580 Betulia (1950 KA),2461000.5,2.195,0.4876,52.19,62.22,159.72,80.27,1.125
7,1620 Geographos (1951 RA),2461000.5,1.246,0.3355,13.34,337.14,277.02,212.92,0.828
8,1627 Ivar (1929 SH),2461000.5,1.863,0.3973,8.46,133.07,167.84,311.46,1.123
9,1685 Toro (1948 OA),2461000.5,1.368,0.4360,9.38,274.21,127.28,82.68,0.771


In [27]:
# Numbered periodic comets, excluding fragments
r = requests.get(SBDB_URL, params={
    "fields": "full_name,a,e,q",
    "sb-kind": "c",
    "sb-ns": "n",
    "sb-xfrag": 1,
    "limit": 5
})
pd.DataFrame(r.json()["data"], columns=r.json()["fields"])

,full_name,a,e,q
0,1P/Halley,17.93,0.9679,0.575
1,2P/Encke,2.22,0.8477,0.338
2,3D/Biela,3.535,0.7513,0.879
3,4P/Faye,3.798,0.5845,1.578
4,5D/Brorsen,3.101,0.8098,0.590


---

## 6. References

- [SBDB Query API](https://ssd-api.jpl.nasa.gov/doc/sbdb_query.html)
- [SBDB Filter parameters](https://ssd-api.jpl.nasa.gov/doc/sbdb_filter.html)
- [JPL Small-Body Database](https://ssd.jpl.nasa.gov/sbdb.cgi)